# NB05: Taxonomic Bias Analysis

Compare taxonomic composition of cultured vs MAG cohorts to determine whether
functional bias is driven by taxonomic sampling gaps (phylum-level absence)
or within-lineage cultivation selection.

**Requires**: NB01 (`data/mag_metadata.tsv`), NB03 (`data/cultured_genome_metadata.tsv`),
NB04 (`data/ko_cultivation_coverage_full.tsv`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

## 1. Load Metadata

In [ ]:
cult_meta = pd.read_csv(DATA_DIR / 'cultured_genome_metadata.tsv', sep='\t')
mag_meta = pd.read_csv(DATA_DIR / 'mag_metadata.tsv', sep='\t')

print(f'Cultured: {len(cult_meta)} genomes')
print(f'MAGs: {len(mag_meta)} genomes')

# Parse taxonomy to phylum level for both
# Cultured: taxon_name from ENIGMA Genome Depot
# MAGs: taxonomy from sdt_bin (format varies — may be GTDB-style or simple name)

print(f'\nCultured taxonomy sample:')
print(cult_meta['taxon_name'].head(10).to_list())
print(f'\nMAG taxonomy sample:')
if 'taxonomy' in mag_meta.columns:
    print(mag_meta['taxonomy'].dropna().head(10).to_list())

## 2. Phylum-Level Comparison

In [ ]:
def extract_phylum(tax_str):
    """Extract phylum from various taxonomy string formats."""
    if pd.isna(tax_str):
        return 'Unknown'
    s = str(tax_str)
    # GTDB format: d__Bacteria;p__Proteobacteria;...
    if 'p__' in s:
        for part in s.split(';'):
            if part.strip().startswith('p__'):
                return part.strip().replace('p__', '')
    # Simple genus/species — look up common mappings or return as-is
    return s.split(';')[0] if ';' in s else s

cult_meta['phylum'] = cult_meta['taxon_name'].apply(extract_phylum)
mag_meta['phylum'] = mag_meta['taxonomy'].apply(extract_phylum)

cult_phyla = cult_meta['phylum'].value_counts().rename('cultured')
mag_phyla = mag_meta['phylum'].value_counts().rename('mag')

phyla = pd.DataFrame({'cultured': cult_phyla, 'mag': mag_phyla}).fillna(0).astype(int)
phyla['cult_frac'] = phyla['cultured'] / phyla['cultured'].sum()
phyla['mag_frac'] = phyla['mag'] / phyla['mag'].sum()
phyla = phyla.sort_values('cultured', ascending=False)

print(f'Phylum distribution:')
print(phyla.head(20).to_string())

# Phyla unique to each cohort
cult_only_phyla = phyla[phyla['mag'] == 0].index.tolist()
mag_only_phyla = phyla[phyla['cultured'] == 0].index.tolist()
print(f'\nPhyla only in cultured: {cult_only_phyla}')
print(f'Phyla only in MAGs: {mag_only_phyla}')

## 3. Relative Abundance Comparison Plot

In [ ]:
# Top phyla bar chart
top_phyla = phyla.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(top_phyla))
w = 0.35
ax.barh(x + w/2, top_phyla['cult_frac'], w, label='Cultured', color='#d62728')
ax.barh(x - w/2, top_phyla['mag_frac'], w, label='MAG', color='#1f77b4')
ax.set_yticks(x)
ax.set_yticklabels(top_phyla.index)
ax.set_xlabel('Fraction of cohort')
ax.set_title('Phylum Composition: Cultured vs MAG Cohorts')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / 'phylum_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Genome Size Comparison

In [ ]:
if 'genome_size' in cult_meta.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    cult_sizes = cult_meta['genome_size'].dropna() / 1e6  # Mbp
    ax.hist(cult_sizes, bins=50, alpha=0.6, label=f'Cultured (n={len(cult_sizes)})', color='#d62728')
    
    # MAG sizes would need to be computed from FASTA files or bin metadata
    # For now, plot cultured only and note the gap
    ax.set_xlabel('Genome size (Mbp)')
    ax.set_ylabel('Count')
    ax.set_title('Genome Size Distribution')
    ax.legend()
    ax.axvline(1.5, ls='--', c='gray', lw=1, label='CPR threshold (1.5 Mbp)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'genome_size_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    small_genomes = cult_sizes[cult_sizes < 1.5]
    print(f'Cultured genomes < 1.5 Mbp (CPR-size): {len(small_genomes)}/{len(cult_sizes)} ({len(small_genomes)/len(cult_sizes)*100:.1f}%)')

## 5. Functional Bias × Taxonomy

In [ ]:
# For MAG-only phyla, what KOs are they carrying that the cultured collection misses?
# Load KO profiles and cross-reference

mag_ko = pd.read_csv(DATA_DIR / 'mag_ko_profiles.tsv', sep='\t')
cult_ko = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')
coverage = pd.read_csv(DATA_DIR / 'ko_cultivation_coverage_full.tsv', sep='\t', index_col=0)

# KOs from MAG-only phyla
mag_only_phyla_genomes = mag_meta[mag_meta['phylum'].isin(mag_only_phyla)]['bin_name']
if len(mag_only_phyla_genomes) > 0:
    mag_only_kos = mag_ko[mag_ko['genome_id'].isin(mag_only_phyla_genomes)]['ko_id'].unique()
    exclusive_kos = set(mag_only_kos) - set(cult_ko['ko_id'].unique())
    print(f'MAG-only phyla contribute {len(mag_only_phyla_genomes)} genomes')
    print(f'  {len(mag_only_kos)} KOs total, {len(exclusive_kos)} exclusive to MAGs')
else:
    print('No MAG-only phyla detected (taxonomy parsing may need adjustment)')